# COOP-UAV-S — train the learned cooperation (WTA) policy on Colab

Trains the MAPPO weapon-target-assignment policy (`docs/MARL.md`) within a fixed wall-clock budget.

**Runtime:** use a **CPU runtime** (Runtime → Change runtime type → **CPU**; pick *High-RAM* on Colab Pro for more vCPUs). A GPU gives **no** benefit here — the bottleneck is the Python battle-sim (env stepping), not the small network, so we parallelise across vCPUs instead.

**Budget:** the `--minutes` flag stops training and checkpoints when the wall clock is hit, so a 1-hour cap is safe. Checkpoints are also written periodically, so whatever you get is usable.

> Honest expectation: ~1 h on Colab's 2–8 vCPUs is a *partial* training run — enough to show the learning trend and produce a working `policy.pt`, not to fully converge past the strong classical baseline. The full run belongs on the many-core box.

### 1. Get the code

In [ ]:
BRANCH = "claude/cap-sentinels-marl-wta"
!git clone --branch {BRANCH} --depth 1 https://github.com/Alexanderis1/COOP-UAV-S.git
%cd COOP-UAV-S

### 2. Install the training extras
Uses Colab's preinstalled PyTorch (CPU is fine) and adds gymnasium / pettingzoo / tensorboard + the sim deps.

In [ ]:
!pip install -q -e ".[train]"
import torch, multiprocessing
print("torch", torch.__version__, "| vCPUs:", multiprocessing.cpu_count())

### 3. Train (wall-clock budget)
One env worker per vCPU; `OMP_NUM_THREADS=1` so the workers don't oversubscribe. Edit `--minutes` to your cap (55 leaves room for install + eval within 1 h).

In [ ]:
import os, multiprocessing
N = multiprocessing.cpu_count()
print(f"using {N} parallel env workers")
!OMP_NUM_THREADS=1 python -m coopuavs train scenarios/high_diver_raid.yaml \
    --n-envs {N} --minutes 55 --rollout 64 --horizon 160 --out runs/marl

### 4. Learning curve

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("runs/marl/train_log.csv")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(df.env_steps, df.mean_ep_return)
ax[0].set(xlabel="env steps", ylabel="mean episode return", title="MAPPO learning curve"); ax[0].grid(True)
ax[1].plot(df.env_steps, df.policy_loss, label="policy")
ax[1].plot(df.env_steps, df.value_loss, label="value")
ax[1].set(xlabel="env steps", title="losses"); ax[1].legend(); ax[1].grid(True)
plt.show(); print(df.tail())

### 5. A/B vs the classical greedy allocator
Same scenario (CAP sentinels included): does the policy convert the sentinels' time margin into fewer armed leakers / less collateral than greedy?

In [ ]:
!python -m coopuavs eval scenarios/high_diver_raid.yaml --policy runs/marl/policy.pt -n 10

### 6. Download the trained policy
Then deploy it in any scenario with `base_station: {allocator: learned, policy: policy.pt}`.

In [ ]:
from google.colab import files
files.download("runs/marl/policy.pt")